In [1]:
# Import of librarys
import hdbscan #For clustering
import opensmile #For feature extraction
import json
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.cluster import k_means
import torch
import torchaudio
from tqdm import tqdm
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor

In [2]:
data_jsonl = "/home/ashley-bravo/SLAM-LLM/examples/asr_librispeech/data/cv_10h/cv_test.jsonl"
audio_files = "/home/ashley-bravo/datasets/common_voice_22_storage"
output_file = "/home/ashley-bravo/clustering/file_cv_test.csv"

def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"Invalid line {line_num} in {path}: {e}")
    return records

data_records = load_jsonl(data_jsonl)

df_final = pd.DataFrame(data_records)

df_final.to_csv(output_file, index=False)
print(f"\nFile saved to: {output_file}")
print(f"Total of records: {len(df_final)}")


File saved to: /home/ashley-bravo/clustering/file_cv_test.csv
Total of records: 6384


In [5]:
clustering_file = "/home/ashley-bravo/clustering/file_cv_test.csv"
output_parquet = "/home/ashley-bravo/clustering/features_cv200_wav2vec_v2.parquet"
errors_file = "/home/ashley-bravo/clustering/extraction_errors.csv"
checkpoint_dir = "/home/ashley-bravo/clustering/checkpoints"

In [3]:
!nvidia-smi

Mon Jul 27 17:33:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 610.43.02              KMD Version: 610.43.02     CUDA UMD Version: 13.3     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L40S                    On  |   00000000:03:00.0 Off |                    0 |
| N/A   67C    P0            310W /  350W |   21027MiB /  46068MiB |    100%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [13]:
import os
import torch
import torchaudio
import pandas as pd
from tqdm import tqdm
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2Model

# ============================
# Configuración
# ============================

MODEL_NAME = "facebook/wav2vec2-base-960h"
TARGET_SAMPLE_RATE = 16000
BATCH_SIZE = 8
CHECKPOINT_EVERY_BATCHES = 800

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {DEVICE}")

# Crear carpeta de checkpoints
os.makedirs(checkpoint_dir, exist_ok=True)
print("Checkpoint dir:", checkpoint_dir)

# ============================
# Función para cargar audio
# ============================

def load_audio(path, target_sr=TARGET_SAMPLE_RATE):
    waveform, sr = torchaudio.load(path)

    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    if sr != target_sr:
        resampler = torchaudio.transforms.Resample(
            orig_freq=sr,
            new_freq=target_sr,
        )
        waveform = resampler(waveform)

    return waveform.squeeze(0)

# ============================
# Leer manifest
# ============================

df_manifest = pd.read_csv(clustering_file)
print(f"Audios a procesar: {len(df_manifest)}")

print(f"Cargando modelo {MODEL_NAME}...")
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)
model = Wav2Vec2Model.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

# ============================
# Reanudar checkpoints
# ============================

processed_keys = set()

existing_checkpoints = sorted(
    f for f in os.listdir(checkpoint_dir)
    if f.endswith(".parquet")
)

if existing_checkpoints:

    print(f"Reanudando desde {len(existing_checkpoints)} checkpoints...")

    for ckpt in existing_checkpoints:
        part = pd.read_parquet(os.path.join(checkpoint_dir, ckpt))
        processed_keys.update(part["key"].tolist())

    print(f"{len(processed_keys)} audios ya procesados.")

df_pending = df_manifest[
    ~df_manifest["key"].isin(processed_keys)
].copy()

print(f"Audios pendientes: {len(df_pending)}")

rows = df_pending.to_dict("records")

errors = []
results_buffer = []

ckpt_idx = len(existing_checkpoints)
batch_count = 0

# ============================
# Procesamiento por batch
# ============================

def process_batch(batch_rows):

    valid_rows = []
    waveforms = []

    local_errors = []

    for row in batch_rows:

        try:

            wav = load_audio(row["source"])

            if wav.numel() == 0:
                raise ValueError("Audio vacío")

            valid_rows.append(row)
            waveforms.append(wav.numpy())

        except Exception as e:
            print(f"\nERROR en {row['source']}")
            print(type(e).__name__, e)
            local_errors.append({
                "key": row["key"],
                "error": str(e)
            })

    if len(waveforms) == 0:
        return [], local_errors

    inputs = feature_extractor(
        waveforms,
        sampling_rate=TARGET_SAMPLE_RATE,
        return_tensors="pt",
        padding=True,
    )

    input_values = inputs.input_values.to(DEVICE)

    attention_mask = inputs.get("attention_mask")

    if attention_mask is not None:
        attention_mask = attention_mask.to(DEVICE)

    with torch.no_grad():

        outputs = model(
            input_values,
            attention_mask=attention_mask,
        )

        hidden = outputs.last_hidden_state

    # ==========================================
    # Masked mean + std pooling
    # ==========================================

    if attention_mask is not None:

        output_lengths = model._get_feat_extract_output_lengths(
            attention_mask.sum(-1)
        )

        B, T, D = hidden.shape

        hidden_mask = torch.zeros(
            (B, T),
            dtype=torch.bool,
            device=hidden.device,
        )

        for i, L in enumerate(output_lengths):
            hidden_mask[i, :L] = True

        hidden_mask = hidden_mask.unsqueeze(-1)

    else:

        hidden_mask = torch.ones(
            hidden.shape[:2],
            dtype=torch.bool,
            device=hidden.device,
        ).unsqueeze(-1)

    mask = hidden_mask.float()

    lengths = mask.sum(dim=1)

    mean_pooled = (hidden * mask).sum(dim=1) / lengths

    variance = (
        ((hidden - mean_pooled.unsqueeze(1)) ** 2) * mask
    ).sum(dim=1) / lengths

    std_pooled = torch.sqrt(variance + 1e-9)

    mean_pooled = mean_pooled.cpu().numpy()
    std_pooled = std_pooled.cpu().numpy()

    batch_results = []

    for i, row in enumerate(valid_rows):

        feat = {
            "key": row["key"],
            "source": row["source"],
            "target": row["target"],
        }

        for d in range(mean_pooled.shape[1]):
            feat[f"mean_{d}"] = mean_pooled[i, d]

        for d in range(std_pooled.shape[1]):
            feat[f"std_{d}"] = std_pooled[i, d]

        batch_results.append(feat)

    return batch_results, local_errors

# ============================
# Extracción
# ============================

with tqdm(total=len(rows)) as pbar:

    for i in range(0, len(rows), BATCH_SIZE):

        batch_rows = rows[i:i+BATCH_SIZE]

        batch_results, batch_errors = process_batch(batch_rows)

        results_buffer.extend(batch_results)
        errors.extend(batch_errors)

        batch_count += 1

        pbar.update(len(batch_rows))

        if batch_count >= CHECKPOINT_EVERY_BATCHES and len(results_buffer):

            ckpt_df = pd.DataFrame(results_buffer)

            ckpt_path = os.path.join(
                checkpoint_dir,
                f"ckpt_{ckpt_idx:04d}.parquet",
            )

            print(f"Guardando {ckpt_path}")

            ckpt_df.to_parquet(
                ckpt_path,
                index=False,
            )

            print("Existe:", os.path.exists(ckpt_path))

            ckpt_idx += 1
            batch_count = 0
            results_buffer = []

# ============================
# Guardar último checkpoint
# ============================

if len(results_buffer):

    ckpt_df = pd.DataFrame(results_buffer)

    ckpt_path = os.path.join(
        checkpoint_dir,
        f"ckpt_{ckpt_idx:04d}.parquet",
    )

    print(f"Guardando último checkpoint: {ckpt_path}")

    ckpt_df.to_parquet(
        ckpt_path,
        index=False,
    )

    print("Existe:", os.path.exists(ckpt_path))

# ============================
# Consolidar
# ============================

all_checkpoints = sorted(
    f for f in os.listdir(checkpoint_dir)
    if f.endswith(".parquet")
)

print("\nCheckpoints encontrados:")
print(all_checkpoints)

if len(all_checkpoints) == 0:
    raise RuntimeError(
        f"No se encontró ningún checkpoint en {checkpoint_dir}"
    )

all_dfs = [
    pd.read_parquet(os.path.join(checkpoint_dir, f))
    for f in all_checkpoints
]

df_final = (
    pd.concat(all_dfs, ignore_index=True)
      .drop_duplicates(subset="key")
)

df_final.to_parquet(
    output_parquet,
    index=False,
)

print(f"\nEmbeddings guardados en {output_parquet}")
print(f"Filas finales: {len(df_final)}")

# ============================
# Guardar errores
# ============================

if len(errors):

    pd.DataFrame(errors).to_csv(
        errors_file,
        index=False,
    )

    print(f"Errores: {len(errors)}")

Usando dispositivo: cuda
Checkpoint dir: /home/ashley-bravo/clustering/checkpoints
Audios a procesar: 6384
Cargando modelo facebook/wav2vec2-base-960h...


Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Audios pendientes: 6384


100%|██████████| 6384/6384 [01:07<00:00, 94.02it/s] 


Guardando último checkpoint: /home/ashley-bravo/clustering/checkpoints/ckpt_0000.parquet
Existe: True

Checkpoints encontrados:
['ckpt_0000.parquet']

Embeddings guardados en /home/ashley-bravo/clustering/features_cv200_wav2vec_v2.parquet
Filas finales: 6384


In [9]:
!pip install transformers accelerate

In [14]:
clustering_file = "/home/ashley-bravo/clustering/file_cv_test.csv"
output_parquet = "/home/ashley-bravo/clustering/features_cv200_whisper_v2.parquet"
errors_file = "/home/ashley-bravo/clustering/extraction_errors.csv"
checkpoint_dir = "/home/ashley-bravo/clustering/checkpoints"

In [15]:
import os
import torch
import torchaudio
import pandas as pd
import numpy as np
from tqdm import tqdm
from transformers import WhisperProcessor, WhisperModel

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {DEVICE}")

MODEL_NAME = "openai/whisper-large-v3"
processor = WhisperProcessor.from_pretrained(MODEL_NAME)

if DEVICE.type == "cuda":
    model = WhisperModel.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16
    )
else:
    model = WhisperModel.from_pretrained(
        MODEL_NAME
    )


model = model.to(DEVICE)
model.eval()

TARGET_SAMPLE_RATE = 16000
BATCH_SIZE = 8
CHECKPOINT_EVERY_BATCHES = 2000

def load_audio(path, target_sr=TARGET_SAMPLE_RATE):
    """
    Carga wav, convierte a mono y resamplea a 16 kHz.
    Devuelve tensor 1D [T]
    """
    waveform, sr = torchaudio.load(path)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(
            dim=0,
            keepdim=True
        )

    # resample
    if sr != target_sr:

        resampler = torchaudio.transforms.Resample(
            orig_freq=sr,
            new_freq=target_sr
        )

        waveform = resampler(waveform)


    return waveform.squeeze(0)

df_manifest = pd.read_csv(clustering_file)
print(
    f"Audios totales: {len(df_manifest)}"
)

processed_keys = set()
existing_checkpoints = sorted(
    f for f in os.listdir(checkpoint_dir)
    if f.endswith(".parquet")
)
if existing_checkpoints:
    print(f"Encontrados {len(existing_checkpoints)} checkpoints")
    for ckpt in existing_checkpoints:
        part = pd.read_parquet(os.path.join(checkpoint_dir,ckpt))
        processed_keys.update(
            part["key"].tolist()
        )


    print(
        f"Audios ya procesados: {len(processed_keys)}"
    )
df_pending = df_manifest[
    ~df_manifest["key"].isin(processed_keys)
].copy()
print(
    f"Pendientes: {len(df_pending)}"
)
rows = df_pending.to_dict("records")
errors = []
results_buffer = []
ckpt_idx = len(existing_checkpoints)
batch_count = 0

def process_batch(batch_rows):

    valid_rows = []
    waveforms = []
    local_errors = []


    # cargar audios

    for row in batch_rows:
        try:
            wav = load_audio(row["source"])
            if wav.numel() == 0:
                raise ValueError(
                    "audio vacío"
                )
            valid_rows.append(row)
            waveforms.append(wav.numpy())
        except Exception as e:
            local_errors.append({"key": row["key"],"error": f"{type(e).__name__}: {e}"})
    if len(waveforms) == 0:

        return [], local_errors

    inputs = processor(
        waveforms,
        sampling_rate=TARGET_SAMPLE_RATE,
        return_tensors="pt",
        padding="max_length",
        max_length=480000,     # 30 segundos @ 16kHz
        truncation=True
    )
    input_features = inputs.input_features.to(DEVICE,dtype=model.dtype)
    with torch.no_grad():
        encoder_outputs = model.encoder(input_features=input_features)
        hidden = encoder_outputs.last_hidden_state

    mean_embedding = hidden.mean(dim=1)
    std_embedding = hidden.std(dim=1)
    embeddings = torch.cat([mean_embedding,std_embedding],dim=1)
    embeddings = embeddings.cpu().float().numpy()

    batch_results = []
    for i, row in enumerate(valid_rows):
        feat_dict = {
            "key": row["key"],"source": row["source"],"target": row["target"]
        }


        for d in range(
            embeddings.shape[1]
        ):

            feat_dict[
                f"emb_{d}"
            ] = embeddings[i,d]


        batch_results.append(
            feat_dict
        )


    return batch_results, local_errors

with tqdm(
    total=len(rows),
    desc="Extrayendo embeddings Whisper"
) as pbar:
    for i in range(
        0,
        len(rows),
        BATCH_SIZE
    ):
        batch_rows = rows[
            i:i+BATCH_SIZE
        ]
        batch_results, batch_errors = process_batch(
            batch_rows
        )
        results_buffer.extend(
            batch_results
        )

        errors.extend(
            batch_errors
        )
        pbar.update(
            len(batch_rows)
        )
        batch_count += 1
        # checkpoint

        if (
            batch_count >= CHECKPOINT_EVERY_BATCHES
            and results_buffer
        ):
            ckpt_df = pd.DataFrame(
                results_buffer
            )
            ckpt_path = os.path.join(
                checkpoint_dir,
                f"ckpt_{ckpt_idx:04d}.parquet"
            )
            ckpt_df.to_parquet(
                ckpt_path,
                index=False
            )
            print(f"Checkpoint guardado: {ckpt_path}")
            ckpt_idx += 1
            results_buffer = []
            batch_count = 0


if results_buffer:

    ckpt_df = pd.DataFrame(
        results_buffer
    )
    ckpt_path = os.path.join(
        checkpoint_dir,
        f"ckpt_{ckpt_idx:04d}.parquet"
    )
    ckpt_df.to_parquet(
        ckpt_path,
        index=False
    )

all_checkpoints = sorted(
    f for f in os.listdir(checkpoint_dir)
    if f.endswith(".parquet")
)
all_dfs = [pd.read_parquet( os.path.join( checkpoint_dir,  f))
    for f in all_checkpoints
]

df_final = pd.concat(all_dfs,ignore_index=True).drop_duplicates(subset="key")
#df_final = pd.merge(df_features,df_manifest[[ "key", "source", "target"]],on="key",how="inner")
df_final.to_parquet(output_parquet,index=False)
print("\nEmbeddings guardados:")
print(output_parquet)
print(f"Total filas: {len(df_final)}")
#print(f"Dimensión embedding: {df_features.shape[1]-1}")
if errors:
    pd.DataFrame(errors).to_csv(
        errors_file,
        index=False
    )
    print(
        f"[WARN] {len(errors)} audios fallaron"
    )

Usando dispositivo: cuda


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

Audios totales: 6384
Pendientes: 6384


Extrayendo embeddings Whisper: 100%|██████████| 6384/6384 [02:28<00:00, 42.87it/s]



Embeddings guardados:
/home/ashley-bravo/clustering/features_cv200_whisper_v2.parquet
Total filas: 6384


### Clustering GMM de Wav2Vec

In [21]:
import json

def save_jsonl(df_subset, output_path):
    with open(output_path, "w", encoding="utf-8") as f:
        for row in df_subset.itertuples(index=False):

            sample = {
                "key": row.key,
                "source": row.source,
                "target": row.target
            }

            json.dump(sample, f, ensure_ascii=False)
            f.write("\n")

import soundfile as sf




In [17]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler

df = pd.read_parquet("/home/ashley-bravo/clustering/features_cv200_wav2vec_v2.parquet")

non_feature_cols = ["key", "source", "target"]
feature_cols = [c for c in df.columns if c not in non_feature_cols]
X = df[feature_cols].values

X_scaled = StandardScaler().fit_transform(X)

In [22]:
import numpy as np
import pandas as pd

from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

gmm = GaussianMixture(n_components=14,covariance_type="full",random_state=42,reg_covar=1e-6)
pca = PCA(n_components=32)
X_pca = pca.fit_transform(X_scaled)
gmm.fit(X_pca)
cluster_probs = gmm.predict_proba(X_pca)
print(cluster_probs.shape)
for i in range(14):
    df[f"cluster_{i}_weight"] = cluster_probs[:,i]

TOP_K = 3

top_indices = np.argsort(cluster_probs,axis=1)[:, -TOP_K:]
soft_weights = np.zeros_like(cluster_probs)
for i,row in enumerate(top_indices):
    for cluster_id in row:
        soft_weights[i,cluster_id] = cluster_probs[i,cluster_id]

soft_weights = (soft_weights /soft_weights.sum(axis=1, keepdims=True))

for cluster_id in range(14):
    df_temp = df.copy()
    df_temp["sampling_weight"] = soft_weights[:,cluster_id]
    save_jsonl(df_temp,f"soft_cluster_{cluster_id}.jsonl")


(6384, 14)


In [23]:
df = pd.read_parquet("/home/ashley-bravo/clustering/features_cv200_whisper_v2.parquet")

non_feature_cols = ["key", "source", "target"]
feature_cols = [c for c in df.columns if c not in non_feature_cols]
X = df[feature_cols].values

X_scaled = StandardScaler().fit_transform(X)

gmm = GaussianMixture(n_components=14,covariance_type="full",random_state=42,reg_covar=1e-6)
pca = PCA(n_components=30)
X_pca = pca.fit_transform(X_scaled)
gmm.fit(X_pca)
cluster_probs = gmm.predict_proba(X_pca)
print(cluster_probs.shape)
for i in range(14):
    df[f"cluster_{i}_weight"] = cluster_probs[:,i]

TOP_K = 3

top_indices = np.argsort(cluster_probs,axis=1)[:, -TOP_K:]
soft_weights = np.zeros_like(cluster_probs)
for i,row in enumerate(top_indices):
    for cluster_id in row:
        soft_weights[i,cluster_id] = cluster_probs[i,cluster_id]

soft_weights = (soft_weights /soft_weights.sum(axis=1, keepdims=True))

for cluster_id in range(14):
    df_temp = df.copy()
    df_temp["sampling_weight"] = soft_weights[:,cluster_id]
    save_jsonl(df_temp,f"soft_cluster_whisper_{cluster_id}.jsonl")

(6384, 14)


# Error Analysis

In [3]:
def load_transcriptions(path):
    transcripts = {}

    with open(path, "r", encoding="utf-8") as fin:
        for line in fin:
            if "\t" not in line:
                continue

            key, texto = line.strip().split("\t", 1)
            transcripts[key] = texto.lower()

    return transcripts

In [ ]:
baseline = load_transcriptions("/home/ashley-bravo/outputs/model_merging/base/asr_epoch_2_step_2251/decode_pred")
merged = load_transcriptions("/home/ashley-bravo/outputs/model_merging/wav2vec_clustering/merged/Linear/fairspeech_pred")
reference = load_transcriptions("/home/ashley-bravo/outputs/model_merging/base/asr_epoch_2_step_2251/decode_gt")


/home/ashley-bravo/outputs/wav2vec_clustering/merged/Linear_exp/inverse_hours/fairspeech_pred

In [6]:
print(list(baseline.keys())[:5])
print(list(merged.keys())[:5])
print(list(reference.keys())[:5])

['fs_ca1920051b7fdc279c45b5211b261cf0_ASR', 'fs_61b469ebfd9e2ee5ec3295b23d282ec4_ASR', 'fs_eb7e37f0cf0c1b5e3587f2ee6a4bd3ba_ASR', 'fs_d1354340600029ae49d0efac5d1dcb14_ASR', 'fs_46b92c6d41ef7b63d1f0ccb40bb1d3c5_ASR']
['fs_ca1920051b7fdc279c45b5211b261cf0_ASR', 'fs_61b469ebfd9e2ee5ec3295b23d282ec4_ASR', 'fs_eb7e37f0cf0c1b5e3587f2ee6a4bd3ba_ASR', 'fs_d1354340600029ae49d0efac5d1dcb14_ASR', 'fs_46b92c6d41ef7b63d1f0ccb40bb1d3c5_ASR']
['fs_ca1920051b7fdc279c45b5211b261cf0_ASR', 'fs_61b469ebfd9e2ee5ec3295b23d282ec4_ASR', 'fs_eb7e37f0cf0c1b5e3587f2ee6a4bd3ba_ASR', 'fs_d1354340600029ae49d0efac5d1dcb14_ASR', 'fs_46b92c6d41ef7b63d1f0ccb40bb1d3c5_ASR']


In [10]:
key = "fs_ca1920051b7fdc279c45b5211b261cf0_ASR"

print(baseline[key])
print(merged[key])
print(reference[key])

print(type(baseline[key]))
print(type(merged[key]))
print(type(reference[key]))

hey facebook answer the call
hey facebook answer the call
hey facebook answer the call
<class 'str'>
<class 'str'>
<class 'str'>


In [14]:
result = analyze_changes(
    baseline[key],
    merged[key],
    reference[key]
)

print(result)

{'before_S': 0, 'before_D': 0, 'before_I': 0, 'after_S': 0, 'after_D': 0, 'after_I': 0, 'before_WER': 0.0, 'after_WER': 0.0}


In [13]:
from jiwer import process_words


def analyze_changes(ref, before, after):

    before_result = process_words(ref, before)
    after_result = process_words(ref, after)

    return {
        "before_S": before_result.substitutions,
        "before_D": before_result.deletions,
        "before_I": before_result.insertions,

        "after_S": after_result.substitutions,
        "after_D": after_result.deletions,
        "after_I": after_result.insertions,

        "before_WER": before_result.wer,
        "after_WER": after_result.wer
    }

In [5]:
def compare_alignment(ref, before, after):

    from jiwer import process_words

    b = process_words(ref, before)
    a = process_words(ref, after)

    improvements = []

    for b_chunk, a_chunk in zip(
        b.alignments[0],
        a.alignments[0]
    ):
        pass

In [16]:
print("Linear merging")
all_results = []

for key in reference.keys():

    result = analyze_changes(
        reference[key],
        baseline[key],
        merged[key]
    )

    result["id"] = key
    all_results.append(result)

Linear merging


In [19]:
import pandas as pd

df_results = pd.DataFrame(all_results)

print(df_results.head())

df_results.to_csv("EA_Linear.csv", index=False)

   before_S  before_D  before_I  after_S  after_D  after_I  before_WER  \
0         0         0         0        0        0        0    0.000000   
1         0         0         0        0        0        0    0.000000   
2         1         0         1        2        0        1    0.222222   
3         2         0         0        2        0        0    0.333333   
4         3         1         0        3        0        0    0.571429   

   after_WER                                       id  
0   0.000000  fs_ca1920051b7fdc279c45b5211b261cf0_ASR  
1   0.000000  fs_61b469ebfd9e2ee5ec3295b23d282ec4_ASR  
2   0.333333  fs_eb7e37f0cf0c1b5e3587f2ee6a4bd3ba_ASR  
3   0.333333  fs_d1354340600029ae49d0efac5d1dcb14_ASR  
4   0.428571  fs_46b92c6d41ef7b63d1f0ccb40bb1d3c5_ASR  


In [20]:
from jiwer import process_words


def extract_word_changes(ref, before, after):

    ref_words = ref.split()
    before_words = before.split()
    after_words = after.split()

    before_result = process_words(ref, before)
    after_result = process_words(ref, after)

    changes = []

    # analizar antes
    for chunk in before_result.alignments[0]:

        ref_part = ref_words[
            chunk.ref_start_idx:chunk.ref_end_idx
        ]

        hyp_part = before_words[
            chunk.hyp_start_idx:chunk.hyp_end_idx
        ]

        changes.append({
            "system": "before",
            "type": chunk.type,
            "reference": ref_part,
            "prediction": hyp_part
        })


    # analizar después
    for chunk in after_result.alignments[0]:

        ref_part = ref_words[
            chunk.ref_start_idx:chunk.ref_end_idx
        ]

        hyp_part = after_words[
            chunk.hyp_start_idx:chunk.hyp_end_idx
        ]

        changes.append({
            "system": "after",
            "type": chunk.type,
            "reference": ref_part,
            "prediction": hyp_part
        })


    return changes

In [22]:
all_alignment_results = []


for key in reference.keys():

    changes = extract_word_changes(
        reference[key],
        baseline[key],
        merged[key]
    )

    all_alignment_results.append({
        "id": key,
        "changes": changes
    })

In [23]:
#Audios were improved

improved = []

for r in all_results:

    if r["after_WER"] < r["before_WER"]:
        improved.append(r)


print(len(improved))

5827


In [24]:
def get_improvements(alignment_results):

    corrected = []

    for item in alignment_results:

        for change in item["changes"]:

            if (
                change["system"] == "before"
                and change["type"] != "equal"
            ):
                corrected.append({
                    "id": item["id"],
                    "error": change
                })

    return corrected

In [25]:
corrected = get_improvements(
    all_alignment_results
)

for x in corrected[:10]:
    print("\nID:", x["id"])
    print(x["error"])


ID: fs_eb7e37f0cf0c1b5e3587f2ee6a4bd3ba_ASR
{'system': 'before', 'type': 'insert', 'reference': [], 'prediction': ['want']}

ID: fs_eb7e37f0cf0c1b5e3587f2ee6a4bd3ba_ASR
{'system': 'before', 'type': 'substitute', 'reference': ['wanna'], 'prediction': ['to']}

ID: fs_d1354340600029ae49d0efac5d1dcb14_ASR
{'system': 'before', 'type': 'substitute', 'reference': ['hey'], 'prediction': ['a']}

ID: fs_d1354340600029ae49d0efac5d1dcb14_ASR
{'system': 'before', 'type': 'substitute', 'reference': ['current'], 'prediction': ['curent']}

ID: fs_46b92c6d41ef7b63d1f0ccb40bb1d3c5_ASR
{'system': 'before', 'type': 'substitute', 'reference': ['hey'], 'prediction': ['a']}

ID: fs_46b92c6d41ef7b63d1f0ccb40bb1d3c5_ASR
{'system': 'before', 'type': 'substitute', 'reference': ['call'], 'prediction': ['post']}

ID: fs_46b92c6d41ef7b63d1f0ccb40bb1d3c5_ASR
{'system': 'before', 'type': 'substitute', 'reference': ['dialed'], 'prediction': ['day']}

ID: fs_46b92c6d41ef7b63d1f0ccb40bb1d3c5_ASR
{'system': 'before', 't

In [26]:
from collections import Counter


error_types = Counter()


for item in all_alignment_results:

    for c in item["changes"]:
        error_types[c["type"]] += 1


print(error_types)

Counter({'equal': 81192, 'substitute': 44049, 'delete': 15860, 'insert': 5450})


In [32]:
from collections import Counter


error_types = Counter()


for item in all_alignment_results2:

    for c in item["changes"]:
        error_types[c["type"]] += 1


print(error_types)

Counter({'equal': 81202, 'substitute': 44018, 'delete': 15759, 'insert': 5461})


# INV HOURS

In [30]:
baseline = load_transcriptions("/home/ashley-bravo/outputs/model_merging/base/asr_epoch_2_step_2251/decode_pred")
merged2 = load_transcriptions("/home/ashley-bravo/outputs/wav2vec_clustering/merged/Linear_exp/inverse_hours/fairspeech_pred")
reference = load_transcriptions("/home/ashley-bravo/outputs/model_merging/base/asr_epoch_2_step_2251/decode_gt")

print("Linear merging")
all_results2 = []

for key in reference.keys():

    result = analyze_changes(
        reference[key],
        baseline[key],
        merged2[key]
    )

    result["id"] = key
    all_results2.append(result)



df_results = pd.DataFrame(all_results2)

print(df_results.head())

df_results.to_csv("EA_Linear_INV.csv", index=False)

Linear merging
   before_S  before_D  before_I  after_S  after_D  after_I  before_WER  \
0         0         0         0        0        0        0    0.000000   
1         0         0         0        0        0        0    0.000000   
2         1         0         1        2        0        1    0.222222   
3         2         0         0        2        0        0    0.333333   
4         3         1         0        3        0        0    0.571429   

   after_WER                                       id  
0   0.000000  fs_ca1920051b7fdc279c45b5211b261cf0_ASR  
1   0.000000  fs_61b469ebfd9e2ee5ec3295b23d282ec4_ASR  
2   0.333333  fs_eb7e37f0cf0c1b5e3587f2ee6a4bd3ba_ASR  
3   0.333333  fs_d1354340600029ae49d0efac5d1dcb14_ASR  
4   0.428571  fs_46b92c6d41ef7b63d1f0ccb40bb1d3c5_ASR  


In [31]:
all_alignment_results2 = []


for key in reference.keys():

    changes = extract_word_changes(
        reference[key],
        baseline[key],
        merged2[key]
    )

    all_alignment_results2.append({
        "id": key,
        "changes": changes
    })


improved2 = []

for r in all_results2:

    if r["after_WER"] < r["before_WER"]:
        improved2.append(r)


print(len(improved2))

6120
